# PETCT-RobustSeg: lung-cancer robustness study (train on all cancers, evaluate on lung)

One notebook that covers **data loading, visualisation, perturbations, augmentation, model, training, evaluation under the frozen perturbations, and result saving**.

**How to use it**
1. Put these files next to this notebook (they come with it): `splits_v2.json`, `perturbations_frozen.json`, `case_index.csv`.
2. Edit **only the first code cell** (`RUN_NAME`, `MODEL_NAME`, `AUGMENT`, `DATA_ROOT`, and the memory settings if your GPU differs). Copy the values for the run you want from the table below.
3. Run all cells top to bottom. Training can be interrupted and resumed (`RESUME = True`); evaluation also resumes.
4. Everything is written to `runs/<RUN_NAME>/`. The last section overlays every run found in `runs/` on the same plots.

**Models to train (copy `RUN_NAME`, `MODEL_NAME` and `AUGMENT` into the first code cell; step-by-step instructions per model, including settings for a faster or slower machine, are in `run_guides/`)**

| `RUN_NAME` | `MODEL_NAME` | `AUGMENT` | purpose |
|---|---|---|---|
| `A_dynunet_noaug` | `dynunet` | `none` | clean baseline |
| `B_dynunet_aug` | `dynunet` | `perturb` | same network with augmentation (isolates the effect of augmentation) |
| `C_segresnet_noaug` | `segresnet` | `none` | architecture comparison against A |
| `D_swinunetr_noaug` | `swinunetr` | `none` | architecture comparison against A (needs `einops`) |
| `E_basic_unet_noaug` | `basic_unet` | `none` | architecture comparison against A |

Change one thing between runs (architecture **or** augmentation) and keep `MAX_EPOCHS` and `VOLUMES_PER_EPOCH` the same.

**What is shared by every model (do not change, otherwise results are not comparable):** the split file, the 15 frozen perturbations, the preprocessing (PET SUV clip [0, 20], CT clip [-100, 250] HU), the metrics, and the evaluation sets (`test_lung`, `test_neg`).

| split | contents | used for |
|---|---|---|
| `train` | 790 patients, all cancer types and lesion-free studies | training |
| `val` | 40 patients (18 lung, 14 other cancers, 8 lesion-free) | checkpoint selection (lung cases first) |
| `test_lung` | 50 lung-cancer patients | headline Dice / lesion recall under perturbations |
| `test_neg` | 20 lesion-free patients | false positives under perturbations |


In [ ]:
import os, sys, json, math, time, hashlib, random, platform, warnings
from pathlib import Path
import numpy as np
import pandas as pd

SMOKE = os.environ.get("PETCT_SMOKE") == "1"      # internal self-test on tiny synthetic data; leave unset

# ================================ EDIT THESE ================================
RUN_NAME   = "A_dynunet_noaug"    # unique name per trained model; results go to runs/<RUN_NAME>/
MODEL_NAME = "dynunet"            # dynunet | segresnet | swinunetr | basic_unet   (see build_model below)
AUGMENT    = "none"               # "none" = clean baseline | "perturb" = blur + noise + PET-dropout augmentation
DATA_ROOT  = r"E:\AI512 Project\PETCT-RobustSeg\data\extracted\PSMA-FDG-PET-CT-Lesions_v2"   # folder that contains imagesTr/ and labelsTr/

# memory / budget (defaults target a 6 GB GPU and 16 GB RAM; lower PATCH or BATCH if you run out of memory)
PATCH = (96, 96, 96)              # training patch and sliding-window size
BATCH = 2
PATCHES_PER_VOLUME = 4            # patches cropped from each loaded volume (amortises the slow .nii.gz read)
FILTERS = (24, 48, 96, 192, 320)  # DynUNet channel widths
MAX_EPOCHS = 60
VOLUMES_PER_EPOCH = 150           # random training volumes per epoch (an "epoch" is a subset, so checkpoints are frequent)
LR = 3e-4
VAL_EVERY = 3                     # epochs between validations
VAL_N = 12                        # validation cases (lung cases first)
LOAD_THREADS = 3                  # volume-loading threads (threads, not processes: safe in notebooks on Windows)
RESUME = True                     # continue from runs/<RUN_NAME>/last.pt if it exists
SEED = 42
# ============================================================================
_expected = f"{MODEL_NAME}_{'noaug' if AUGMENT == 'none' else 'aug'}"
assert RUN_NAME.endswith(_expected), (f"RUN_NAME '{RUN_NAME}' does not match MODEL_NAME / AUGMENT: it should end in '{_expected}' "
                                      "(see the table in the first markdown cell), otherwise results from different models get mixed up")

# ---- shared by every model: do NOT change ----
PET_CLIP = (0.0, 20.0)            # SUV -> [0, 1]
CT_CLIP = (-100.0, 250.0)         # HU  -> [0, 1]
SW_BATCH, SW_OVERLAP = 2, 0.25    # sliding-window inference
PRED_THRESHOLD = 0.5              # lesion probability threshold
LESION_MIN_OVERLAP = 0.10         # a ground-truth lesion counts as detected if >= 10% of its voxels are predicted
EVAL_SPLITS = ("test_lung", "test_neg")
EXPECTED_SPLIT_FP = "03e0439b74b352b7922d6bee526e7a99b7cfc01e75c8b86f2461d6172a93168c"   # fingerprint of splits_v2.json patients
EXPECTED_PERT_FP = "bca490f76304c743171fbe577b989b8ac75305e95a8a8e3e805bcbb528559c51"   # fingerprint of the 15 frozen perturbations
EVAL_MAX_CASES = None             # set e.g. 5 for a quick look; None = all cases
# ----------------------------------------------

CODE_DIR = Path(os.environ.get("PETCT_CODE_DIR", ".")).resolve()
DATA_ROOT = os.environ.get("PETCT_DATA_ROOT", DATA_ROOT)
SPLIT_FILE, FROZEN_FILE, INDEX_FILE = CODE_DIR / "splits_v2.json", CODE_DIR / "perturbations_frozen.json", CODE_DIR / "case_index.csv"
OUT_DIR = CODE_DIR / "runs" / RUN_NAME
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

if SMOKE:   # tiny settings so the whole notebook runs on CPU in a minute
    PATCH, BATCH, PATCHES_PER_VOLUME, FILTERS = (32, 32, 32), 2, 2, (8, 16, 32, 64, 96)
    MAX_EPOCHS, VOLUMES_PER_EPOCH, VAL_EVERY, VAL_N, LOAD_THREADS, SW_BATCH = 2, 4, 1, 2, 2, 1
    EVAL_MAX_CASES = 2
def status(msg):   # one line per epoch / evaluated study in runs/<RUN_NAME>/status.log, for checking on long runs
    with open(OUT_DIR / "status.log", "a") as f:
        f.write(f"{time.strftime('%Y-%m-%d %H:%M:%S')}  {msg}\n")
print(f"run: {RUN_NAME} | model: {MODEL_NAME} | augment: {AUGMENT} | outputs: {OUT_DIR}")

In [ ]:
import torch, torch.nn.functional as F
import monai
from monai.transforms import (Compose, LoadImaged, EnsureChannelFirstd, Orientationd, ScaleIntensityRanged, ConcatItemsd,
                              CopyItemsd, EnsureTyped, SelectItemsd, RandCropByPosNegLabeld, MapTransform, AsDiscreted)
from monai.networks.nets import DynUNet, SegResNet, SwinUNETR, BasicUNet
from monai.losses import DiceCELoss, DeepSupervisionLoss
from monai.inferers import sliding_window_inference
from scipy import ndimage
import matplotlib
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass

warnings.filterwarnings("ignore", category=FutureWarning)                       # harmless library deprecation chatter
warnings.filterwarnings("ignore", message=".*unable to generate class balanced samples.*")   # expected for lesion-free volumes
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
seed_everything(SEED)

ENV = {"python": platform.python_version(), "torch": torch.__version__, "monai": monai.__version__, "numpy": np.__version__,
       "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}
if torch.cuda.is_available():
    ENV["vram_gb"] = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
print("  ".join(f"{k}: {v}" for k, v in ENV.items()))

## 1. Data index and the shared split

`case_index.csv` lists every study (paths relative to `DATA_ROOT`). `splits_v2.json` assigns whole **patients** to splits, so no patient appears in two splits. The cell below refuses to continue if the split file is not the agreed one.

In [ ]:
index = pd.read_csv(INDEX_FILE)
splits = json.load(open(SPLIT_FILE))
split_fp = hashlib.sha256(json.dumps(splits["patients"], sort_keys=True).encode()).hexdigest()
if not SMOKE:
    assert split_fp == EXPECTED_SPLIT_FP, "splits_v2.json is not the agreed split: do not edit or regenerate it"
names = list(splits["patients"])
for i, a in enumerate(names):
    for b in names[i + 1:]:
        assert not set(splits["patients"][a]) & set(splits["patients"][b]), f"patient leakage between {a} and {b}"

index["split"] = index["patient_id"].map({p: s for s, v in splits["patients"].items() for p in v})
index["lesion_positive"] = index["lesion_voxels"] > 0
summary = (index.dropna(subset=["split"]).groupby("split")
           .agg(patients=("patient_id", "nunique"), studies=("case_id", "size"), lesion_positive=("lesion_positive", "sum"))
           .reindex(names))
summary["lung_studies"] = [int(((index.split == s) & (index.diagnosis == "LUNG_CANCER")).sum()) for s in names]
display(summary)

idx = index.set_index("case_id")
def make_records(split_name):
    recs = []
    for cid in splits["cases"][split_name]:
        r = idx.loc[cid]
        recs.append({"case_id": cid, "patient_id": r["patient_id"], "diagnosis": r["diagnosis"],
                     "pet": str(Path(DATA_ROOT) / r["pet"]), "ct": str(Path(DATA_ROOT) / r["ct"]), "mask": str(Path(DATA_ROOT) / r["mask"]),
                     "lesion_voxels": int(r["lesion_voxels"]), "voxel_ml": float(r["dx"] * r["dy"] * r["dz"] / 1000.0),
                     "aspect": float(r["dz"] / r["dx"])})
    return recs

RECORDS = {s: make_records(s) for s in names}
missing = [p for s in RECORDS.values() for r in s for p in (r["pet"], r["ct"], r["mask"]) if not Path(p).exists()]
assert not missing, f"{len(missing)} files not found under DATA_ROOT, e.g. {missing[:2]}"
print("all files found; split fingerprint", split_fp[:12])

## 2. Loading and preprocessing

Each study becomes a 2-channel volume `image = [PET, CT]` scaled to [0, 1] and a binary `mask`, all on the same grid (checked), reoriented to RAS. For evaluation the raw SUV / HU copies are also kept (used for per-lesion features and the body bounding box).

In [ ]:
class CheckSameGridd(MapTransform):
    """Fail loudly if PET, CT and mask do not share one voxel grid (shape + affine).
    4 autoPET FDG lesion-free labels have a flipped qform-only header; an EMPTY mask has no geometry to get wrong, so adopt the image affine."""
    def __init__(self, keys, atol=1e-2):
        super().__init__(keys); self.atol = atol
    def __call__(self, data):
        d = dict(data); ref_key = self.keys[0]; ref = d[ref_key]
        for k in self.keys[1:]:
            same_shape = tuple(ref.shape) == tuple(d[k].shape)
            same_affine = np.allclose(np.asarray(ref.affine), np.asarray(d[k].affine), atol=self.atol)
            if same_shape and not same_affine and k == "mask" and float(d[k].sum()) == 0.0:
                d[k].meta["affine"] = d[ref_key].meta["affine"].clone(); continue
            if not (same_shape and same_affine):
                raise ValueError(f"grid mismatch for {d.get('case_id', '?')}: '{ref_key}' {tuple(ref.shape)} vs '{k}' {tuple(d[k].shape)}")
        return d

def build_tf(keep_raw=False):
    """Full-volume pipeline -> {'image': (2,X,Y,Z), 'mask': (1,X,Y,Z)} (+ 'pet_raw','ct_raw' in SUV / HU if keep_raw)."""
    keys = ["pet", "ct", "mask"]
    t = [LoadImaged(keys=keys, image_only=True, dtype=np.float32), EnsureChannelFirstd(keys=keys),
         CheckSameGridd(keys=keys), Orientationd(keys=keys, axcodes="RAS")]
    if keep_raw:
        t.append(CopyItemsd(keys=["pet", "ct"], names=["pet_raw", "ct_raw"]))
    t += [ScaleIntensityRanged(keys="pet", a_min=PET_CLIP[0], a_max=PET_CLIP[1], b_min=0.0, b_max=1.0, clip=True),
          ScaleIntensityRanged(keys="ct", a_min=CT_CLIP[0], a_max=CT_CLIP[1], b_min=0.0, b_max=1.0, clip=True),
          ConcatItemsd(keys=["pet", "ct"], name="image", dim=0), AsDiscreted(keys="mask", threshold=0.5)]
    keep = ["image", "mask"] + (["pet_raw", "ct_raw"] if keep_raw else [])
    t += [EnsureTyped(keys=keep, dtype=torch.float32, track_meta=False), SelectItemsd(keys=keep + ["case_id"])]
    return Compose(t)

def load_case(rec, keep_raw=False):
    return build_tf(keep_raw)({k: rec[k] for k in ("pet", "ct", "mask", "case_id")})

def prefetch_map(fn, items, workers, ahead=None):
    """Yield fn(item) in order, computed by a small thread pool a few items ahead (hides the slow .nii.gz decode)."""
    ahead = ahead or workers * 2
    items = list(items)
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futs = [ex.submit(fn, it) for it in items[:ahead]]
        nxt = ahead
        for i in range(len(items)):
            res = futs[i].result()
            if nxt < len(items):
                futs.append(ex.submit(fn, items[nxt])); nxt += 1
            futs[i] = None
            yield res
pass

## 3. Visualisation of the data

Axial and coronal slices through the largest lesion, PET/CT fusion, and the PET maximum-intensity projection with the ground-truth lesion outline.

In [ ]:
def _slice_through_lesion(mask):
    m = mask[0].numpy() > 0
    if not m.any():
        return tuple(s // 2 for s in m.shape)
    z = int(m.sum(axis=(0, 1)).argmax())
    cx, cy = ndimage.center_of_mass(m[:, :, z])
    return int(round(cx)), int(round(cy)), z

def _contour(ax, m2d):
    if m2d.any():
        ax.contour(m2d.T.astype(float), levels=[0.5], colors="lime", linewidths=0.9, origin="lower")

def show_case(sample, title="", aspect=1.47, save=None):
    img, msk = sample["image"].numpy(), sample["mask"][0].numpy() > 0
    x, y, z = _slice_through_lesion(sample["mask"])
    pet, ct = img[0], img[1]
    fig, ax = plt.subplots(2, 4, figsize=(15, 8.2))
    views = [("axial", lambda a: a[:, :, z].T, 1.0), ("coronal", lambda a: a[:, y, :].T, aspect)]
    for r, (vn, sl, asp) in enumerate(views):
        ax[r, 0].imshow(sl(pet), cmap="gray_r", vmin=0, vmax=0.4, origin="lower", aspect=asp); ax[r, 0].set_title(f"PET ({vn}), SUV 0-8")
        ax[r, 1].imshow(sl(ct), cmap="gray", vmin=0, vmax=1, origin="lower", aspect=asp); ax[r, 1].set_title("CT, window -100..250 HU")
        ax[r, 2].imshow(sl(ct), cmap="gray", vmin=0, vmax=1, origin="lower", aspect=asp)
        ax[r, 2].imshow(np.ma.masked_less(sl(pet), 0.05), cmap="hot", vmin=0, vmax=0.4, alpha=0.55, origin="lower", aspect=asp); ax[r, 2].set_title("PET/CT fusion")
        _contour(ax[r, 2], sl(msk))
    ax[0, 3].imshow(ct[:, :, z].T, cmap="gray", vmin=0, vmax=1, origin="lower"); _contour(ax[0, 3], msk[:, :, z]); ax[0, 3].set_title("ground-truth lesions (axial)")
    ax[1, 3].imshow(pet.max(axis=1).T, cmap="gray_r", vmin=0, vmax=0.6, origin="lower", aspect=aspect); _contour(ax[1, 3], msk.any(axis=1)); ax[1, 3].set_title("PET MIP (coronal) + lesions")
    for a in ax.ravel():
        a.axis("off")
    fig.suptitle(title); fig.tight_layout()
    if save:
        fig.savefig(FIG_DIR / save, dpi=130)
    plt.show()

def _first(split, diag=None, positive=True):
    for r in RECORDS[split]:
        if (diag is None or r["diagnosis"] == diag) and ((r["lesion_voxels"] > 0) == positive):
            return r

demo_train = _first("train", "LUNG_CANCER"); demo_test = _first("test_lung")
for rec, nm in [(demo_train, "train lung case"), (demo_test, "test lung case")]:
    t0 = time.time(); s = load_case(rec)
    print(f"{nm}: patient {rec['patient_id']} | volume {tuple(s['image'].shape[1:])} | lesion voxels {int(s['mask'].sum()):,} | load {time.time()-t0:.1f}s")
    show_case(s, f"{nm} ({rec['diagnosis']})", rec["aspect"], save=f"data_{nm.split()[0]}.png")
    del s

## 4. The frozen perturbations

Defined **before** any evaluation and frozen in `perturbations_frozen.json` (15 conditions): low-resolution resampling (x1.5, 2, 3, 4; area down-sampling then trilinear back to the original grid), Gaussian noise (sigma 0.05, 0.10, 0.20 on PET, CT or both, in normalised units, not clipped) and PET removal (PET channel set to 0). The ground-truth mask is never touched, so image, prediction and mask stay on the same grid. Noise is seeded from `(case_id, condition)`, so every model sees exactly the same corrupted test images.

These are synthetic image-quality changes; they are not claimed to represent any particular scanner.

In [ ]:
PET, CT = 0, 1
_TARGET = {"pet": (PET,), "ct": (CT,), "both": (PET, CT)}

@dataclass(frozen=True)
class Spec:
    name: str; kind: str; family: str; level: float; target: str = ""
    def to_dict(self):
        return {"name": self.name, "kind": self.kind, "family": self.family, "level": self.level, "target": self.target}

def seed_from(case_id, name):
    return int.from_bytes(hashlib.sha256(f"{case_id}|{name}".encode()).digest()[:8], "little") & 0x7FFFFFFFFFFFFFFF

def low_resolution(image, factor, channels=(PET, CT)):
    out = image.float().clone()
    if factor == 1:
        return out
    spatial = tuple(out.shape[1:]); small = tuple(max(1, int(np.ceil(s / factor))) for s in spatial)
    for c in channels:
        down = F.interpolate(out[c][None, None], size=small, mode="area")
        out[c] = F.interpolate(down, size=spatial, mode="trilinear", align_corners=False)[0, 0]
    return out

def gaussian_noise(image, sigma, target, case_id, name):
    out = image.float().clone()
    if sigma == 0:
        return out
    gen = torch.Generator().manual_seed(seed_from(case_id, name))
    for c in _TARGET[target]:
        out[c] = out[c] + sigma * torch.randn(out[c].shape, generator=gen)
    return out

def remove_pet(image):
    out = image.float().clone(); out[PET] = 0.0
    return out

def apply_spec(spec, image, case_id):
    if spec.kind == "clean":
        return image.float().clone()
    if spec.kind == "lowres":
        return low_resolution(image, spec.level)
    if spec.kind == "noise":
        return gaussian_noise(image, spec.level, spec.target, case_id, spec.name)
    if spec.kind == "pet_removed":
        return remove_pet(image)
    raise ValueError(spec.kind)

frozen = json.load(open(FROZEN_FILE))
SPECS = [Spec(**c) for c in frozen["conditions"]]
fp = hashlib.sha256(json.dumps([s.to_dict() for s in SPECS], sort_keys=True).encode()).hexdigest()
assert fp == frozen["fingerprint"], "perturbation file does not match its own fingerprint"
if not SMOKE:
    assert fp == EXPECTED_PERT_FP, "perturbations_frozen.json is not the agreed set of perturbations"
COND = {s.name: s for s in SPECS}
print(f"{len(SPECS)} frozen conditions, fingerprint {fp[:12]} (verified)")

In [ ]:
def perturbation_gallery(rec, channel=PET, save=None):
    """One axial slice through the largest lesion under all 15 conditions (PET or CT)."""
    s = load_case(rec); _, _, z = _slice_through_lesion(s["mask"]); m2d = s["mask"][0, :, :, z].numpy() > 0
    fig, axs = plt.subplots(3, 5, figsize=(14, 8.6))
    for a, spec in zip(axs.ravel(), SPECS):
        sl = apply_spec(spec, s["image"], rec["case_id"])[channel, :, :, z].numpy()
        a.imshow(sl.T, cmap="gray_r" if channel == PET else "gray", vmin=0, vmax=0.4 if channel == PET else 1, origin="lower")
        _contour(a, m2d); a.set_title(spec.name, fontsize=10); a.axis("off")
    fig.suptitle(f"{'PET' if channel == PET else 'CT'} under the 15 frozen conditions (green = ground-truth lesion)"); fig.tight_layout()
    if save:
        fig.savefig(FIG_DIR / save, dpi=120)
    plt.show()

perturbation_gallery(demo_test, PET, "perturbations_pet.png")
perturbation_gallery(demo_test, CT, "perturbations_ct.png")

## 5. Training-time augmentation

* `AUGMENT = "none"`: only a random left-right flip (a geometric change that does not touch image quality). This is the **clean baseline**.
* `AUGMENT = "perturb"`: the flip plus, per patch, random **blur** (low-resolution factor 1.2-4, p = 0.3), random **noise** (sigma 0.02-0.15 on PET, CT or both, p = 0.4) and **PET dropout** (PET set to 0, p = 0.15). The parameters are drawn from continuous ranges, so they bracket the frozen test levels without copying them exactly. Blur is applied per patch here, but per whole volume at test time.

Both settings use the same random patch sampling: 2/3 of the patches are centred on a lesion, 1/3 on a random location (lesion-free studies give only random patches).

In [ ]:
def augment_patch(img, msk):
    """img (2,x,y,z) in [0,1], msk (1,x,y,z). Returns new tensors."""
    img = img.clone(); msk = msk.clone()
    if random.random() < 0.5:                                  # left-right flip (both models)
        img, msk = img.flip(1), msk.flip(1)
    if AUGMENT == "perturb":
        if random.random() < 0.30:
            img = low_resolution(img, random.uniform(1.2, 4.0))
        if random.random() < 0.40:
            sigma = random.uniform(0.02, 0.15)
            for c in _TARGET[random.choice(["pet", "ct", "both"])]:
                img[c] = img[c] + sigma * torch.randn_like(img[c])
        if random.random() < 0.15:
            img[PET] = 0.0
    return img, msk

import threading
_tls = threading.local()

def train_tf():
    """One transform per loader thread: RandCropByPosNegLabeld stores its crop centres on the object, so a shared
    instance lets another thread change the centres between the image crop and the mask crop."""
    if not hasattr(_tls, "tf"):
        _tls.tf = Compose([build_tf(False),
                           RandCropByPosNegLabeld(keys=["image", "mask"], label_key="mask", spatial_size=PATCH, pos=2, neg=1,
                                                  num_samples=PATCHES_PER_VOLUME)])
    return _tls.tf

def load_train_patches(rec):
    items = train_tf()({k: rec[k] for k in ("pet", "ct", "mask", "case_id")})
    return [augment_patch(it["image"], it["mask"]) for it in items]

# preview: one real training volume, 6 augmentation draws of the same patch
_prev = load_train_patches(_first("train", "LUNG_CANCER"))
base_img, base_msk = _prev[0]
fig, axs = plt.subplots(2, 6, figsize=(16, 5.6)); zc = base_img.shape[3] // 2
for j in range(6):
    im, mk = augment_patch(base_img, base_msk)
    axs[0, j].imshow(im[PET, :, :, zc].T, cmap="gray_r", vmin=0, vmax=0.4, origin="lower"); axs[0, j].set_title(f"PET draw {j+1}", fontsize=9)
    axs[1, j].imshow(im[CT, :, :, zc].T, cmap="gray", vmin=0, vmax=1, origin="lower"); axs[1, j].set_title("CT", fontsize=9)
    for r in (0, 1):
        axs[r, j].axis("off")
fig.suptitle(f"AUGMENT = '{AUGMENT}'"); fig.tight_layout(); fig.savefig(FIG_DIR / "augmentation_preview.png", dpi=120); plt.show()
del _prev

## 6. Model

`dynunet` is MONAI's nnU-Net-style 3D U-Net (instance norm, leaky ReLU, deep supervision), about 11 M parameters at the default widths `FILTERS = (24, 48, 96, 192, 320)` (nnU-Net's default 3D network is about 31 M; the widths are reduced for a 6 GB GPU, and you can raise them if the peak-memory line below leaves room). Call it "nnU-Net-style U-Net" in the report, not "nnU-Net": the official nnU-Net pipeline is not used.

To add another architecture (for example a teammate's model), add a branch to `build_model` that returns `(model, uses_deep_supervision)`. The model must take `(B, 2, X, Y, Z)` and return `(B, 2, X, Y, Z)` logits in eval mode. Nothing else in the notebook changes.

In [ ]:
N_IN, N_CLASSES = 2, 2

def build_model(name=MODEL_NAME):
    if name == "dynunet":
        strides = [[1, 1, 1]] + [[2, 2, 2]] * (len(FILTERS) - 1)
        m = DynUNet(spatial_dims=3, in_channels=N_IN, out_channels=N_CLASSES, kernel_size=[[3, 3, 3]] * len(FILTERS),
                    strides=strides, upsample_kernel_size=strides[1:], filters=list(FILTERS),
                    norm_name=("INSTANCE", {"affine": True}), act_name=("leakyrelu", {"inplace": True, "negative_slope": 0.01}),
                    deep_supervision=True, deep_supr_num=2, res_block=False)
        return m, True
    if name == "segresnet":
        return SegResNet(spatial_dims=3, init_filters=16, in_channels=N_IN, out_channels=N_CLASSES,
                         blocks_down=(1, 2, 2, 4), blocks_up=(1, 1, 1)), False
    if name == "swinunetr":
        try:
            m = SwinUNETR(img_size=PATCH, in_channels=N_IN, out_channels=N_CLASSES, feature_size=24, use_checkpoint=True)
        except TypeError:   # newer MONAI removed img_size
            m = SwinUNETR(in_channels=N_IN, out_channels=N_CLASSES, feature_size=24, use_checkpoint=True)
        return m, False   # use_checkpoint: gradient checkpointing, same result with less GPU memory (6.9 -> 4.6 GB at patch 96, batch 2)
    if name == "basic_unet":
        return BasicUNet(spatial_dims=3, in_channels=N_IN, out_channels=N_CLASSES, features=(16, 32, 64, 128, 256, 16)), False
    raise ValueError(f"unknown MODEL_NAME {name}")

model, DEEP_SUP = build_model()
model = model.to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"{MODEL_NAME}: {n_params/1e6:.1f} M parameters, deep supervision: {DEEP_SUP}")

if DEVICE.type == "cuda":   # peak-memory check with a dummy batch, so an out-of-memory error shows up now rather than after loading data
    torch.cuda.reset_peak_memory_stats()
    model.train()
    x = torch.randn(BATCH, N_IN, *PATCH, device=DEVICE)
    with torch.autocast("cuda", dtype=torch.float16):
        y = model(x)
    y.float().mean().backward(); model.zero_grad(set_to_none=True)
    print(f"peak GPU memory for a training step (batch {BATCH}, patch {PATCH}): {torch.cuda.max_memory_allocated()/1e9:.2f} GB of {ENV['vram_gb']} GB")
    del x, y; torch.cuda.empty_cache()

## 7. Inference and metrics

* **Inference**: sliding window over the body bounding box (air outside the body is skipped; predictions outside the box are zero), Gaussian-weighted, 25% overlap, softmax threshold 0.5. Perturbations are applied to the **whole** volume first, so they match the frozen definition exactly.
* **Dice** (voxel overlap) is computed on lesion-positive studies; it is undefined (NaN) when the ground truth is empty.
* **Lesion recall** (lesion-wise): ground-truth lesions are 26-connected components of the mask; a lesion is *detected* if at least `LESION_MIN_OVERLAP` (10%) of its voxels are predicted. The raw overlap fraction is saved too, so the threshold can be changed later without re-running.
* **False positives**: predicted components that touch no ground-truth lesion (count and volume in ml). Reported on lesion-positive studies and, mainly, on the lesion-free `test_neg` studies.
* **Lesion features** (volume, SUV max/mean, SUV ratio to a surrounding shell, HU contrast to the same shell), computed once per study from the clean raw images, so that "which lesions are lost first" can be analysed later.

In [ ]:
STRUCT26 = np.ones((3, 3, 3), dtype=bool)

def body_mask(ct_hu):
    """Patient body (air outside removed, lungs filled in): HU > -300 with holes filled slice by slice."""
    b = ct_hu > -300
    for z in range(b.shape[2]):
        if b[:, :, z].any():
            b[:, :, z] = ndimage.binary_fill_holes(b[:, :, z])
    return b

def bbox_slices(body, gt, margin=8):
    m = body | gt; sls = []
    for d in range(3):
        nz = np.flatnonzero(m.any(axis=tuple(a for a in range(3) if a != d)))
        lo = max(0, nz[0] - margin) if len(nz) else 0
        hi = min(m.shape[d], nz[-1] + margin + 1) if len(nz) else m.shape[d]
        sls.append(slice(lo, hi))
    return tuple(sls)

@torch.no_grad()
def predict_crop(net, image_crop):
    """image_crop: (2,x,y,z) float32 CPU tensor -> boolean numpy lesion prediction of the same size."""
    net.eval()
    with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
        logits = sliding_window_inference(image_crop[None], roi_size=PATCH, sw_batch_size=SW_BATCH, predictor=net,
                                          overlap=SW_OVERLAP, mode="gaussian", sw_device=DEVICE, device=torch.device("cpu"), progress=False)
    return (torch.softmax(logits.float(), dim=1)[0, 1] > PRED_THRESHOLD).numpy()

def make_gt_pack(gt, voxel_ml):
    lab, n = ndimage.label(gt, structure=STRUCT26)
    return {"gt": gt, "lab": lab, "n": n, "objs": ndimage.find_objects(lab) if n else [],
            "sizes": np.bincount(lab.ravel(), minlength=n + 1)[1:] if n else np.zeros(0, int), "voxel_ml": voxel_ml}

def score_prediction(pred, gp):
    gt = gp["gt"]
    inter, ps, gs = int(np.logical_and(pred, gt).sum()), int(pred.sum()), int(gt.sum())
    lesions = []
    for i, sl in enumerate(gp["objs"], 1):
        size = int(gp["sizes"][i - 1]); ov = int(np.logical_and(pred[sl], gp["lab"][sl] == i).sum())
        lesions.append({"lesion_id": i, "gt_voxels": size, "overlap_voxels": ov, "overlap_frac": ov / size,
                        "detected": int(ov / size >= LESION_MIN_OVERLAP)})
    n_fp, fp_vox = 0, 0
    if ps > 0:
        plab, pn = ndimage.label(pred, structure=STRUCT26)
        hit = np.unique(plab[gt & (plab > 0)]); sizes_p = np.bincount(plab.ravel(), minlength=pn + 1)
        fp_ids = np.setdiff1d(np.arange(1, pn + 1), hit); n_fp, fp_vox = len(fp_ids), int(sizes_p[fp_ids].sum())
    row = {"gt_voxels": gs, "pred_voxels": ps, "dice": 2 * inter / (ps + gs) if gs > 0 else np.nan,
           "voxel_recall": inter / gs if gs > 0 else np.nan, "voxel_precision": inter / ps if ps > 0 else np.nan,
           "n_gt_lesions": len(lesions), "n_detected": int(sum(l["detected"] for l in lesions)),
           "n_fp_components": n_fp, "fp_volume_ml": fp_vox * gp["voxel_ml"], "pred_volume_ml": ps * gp["voxel_ml"]}
    return row, lesions

def lesion_feature_rows(gp, pet_suv, ct_hu, body, margin=10):
    rows = []
    for i, sl in enumerate(gp["objs"], 1):
        sl2 = tuple(slice(max(0, s.start - margin), min(dim, s.stop + margin)) for s, dim in zip(sl, gp["gt"].shape))
        comp = gp["lab"][sl2] == i
        shell = (ndimage.binary_dilation(comp, iterations=8) & ~ndimage.binary_dilation(comp, iterations=2)
                 & ~gp["gt"][sl2] & body[sl2])
        suv, hu = pet_suv[sl2][comp], ct_hu[sl2][comp]
        bg_suv = float(np.median(pet_suv[sl2][shell])) if shell.any() else np.nan
        bg_hu = float(np.median(ct_hu[sl2][shell])) if shell.any() else np.nan
        rows.append({"lesion_id": i, "gt_voxels": int(gp["sizes"][i - 1]), "volume_ml": float(gp["sizes"][i - 1] * gp["voxel_ml"]),
                     "suv_max": float(suv.max()), "suv_mean": float(suv.mean()), "bg_suv": bg_suv,
                     "suv_ratio": float(suv.mean() / bg_suv) if bg_suv and bg_suv > 0 else np.nan,
                     "hu_mean": float(hu.mean()), "bg_hu": bg_hu, "hu_contrast": float(hu.mean() - bg_hu) if bg_hu == bg_hu else np.nan,
                     "z_frac": float((sl[2].start + sl[2].stop) / 2 / gp["gt"].shape[2])})
    return rows

def validate(net, recs):
    """Clean-image validation on lesion-positive cases: mean Dice and pooled lesion recall."""
    dices, det, ngt = [], 0, 0
    for s, rec in zip(prefetch_map(lambda r: load_case(r, keep_raw=True), recs, 2), recs):
        ct = s["ct_raw"][0].numpy(); gt = s["mask"][0].numpy() > 0
        bb = bbox_slices(body_mask(ct), gt)
        pred = predict_crop(net, s["image"][(slice(None),) + bb])
        row, _ = score_prediction(pred, make_gt_pack(gt[bb], rec["voxel_ml"]))
        dices.append(row["dice"]); det += row["n_detected"]; ngt += row["n_gt_lesions"]
    return float(np.nanmean(dices)), det / max(ngt, 1)
pass

## 8. Training

Dice + cross-entropy loss (with deep supervision for `dynunet`), AdamW with a polynomial learning-rate decay, mixed precision on the GPU, patches cropped from volumes loaded by background threads. The best checkpoint (by clean validation Dice on lung cases) is kept in `best.pt`; `last.pt` allows resuming after an interruption.

In [ ]:
train_recs, val_all = RECORDS["train"], RECORDS["val"]
val_recs = ([r for r in val_all if r["diagnosis"] == "LUNG_CANCER" and r["lesion_voxels"] > 0]
            + [r for r in val_all if not (r["diagnosis"] == "LUNG_CANCER" and r["lesion_voxels"] > 0) and r["lesion_voxels"] > 0])[:VAL_N]
print(f"train studies: {len(train_recs)} | validation studies: {len(val_recs)} (lung first)")

base_loss = DiceCELoss(to_onehot_y=True, softmax=True, include_background=False, batch=True, lambda_dice=1.0, lambda_ce=1.0)
loss_fn = DeepSupervisionLoss(base_loss) if DEEP_SUP else base_loss
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
try:
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
except (AttributeError, TypeError):
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

def patch_batches(recs):
    pool = []
    for patches in prefetch_map(load_train_patches, recs, LOAD_THREADS):
        pool.extend(patches)
        while len(pool) >= BATCH * 4:
            take = [pool.pop(random.randrange(len(pool))) for _ in range(BATCH)]
            yield torch.stack([t[0] for t in take]), torch.stack([t[1] for t in take])
    random.shuffle(pool)
    while len(pool) >= BATCH:
        take = [pool.pop() for _ in range(BATCH)]
        yield torch.stack([t[0] for t in take]), torch.stack([t[1] for t in take])

def train_one_epoch(epoch):
    model.train(); losses = []
    recs = random.sample(train_recs, min(VOLUMES_PER_EPOCH, len(train_recs)))
    for xb, yb in patch_batches(recs):
        xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            out = model(xb)
        loss = loss_fn(list(torch.unbind(out.float(), dim=1)) if DEEP_SUP else out.float(), yb)   # deep supervision stacks heads on dim 1
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 12.0)
        scaler.step(opt); scaler.update()
        losses.append(float(loss.detach()))
    return float(np.mean(losses)) if losses else float("nan")

LOG_FILE, BEST_FILE, LAST_FILE = OUT_DIR / "train_log.csv", OUT_DIR / "best.pt", OUT_DIR / "last.pt"
log, start_epoch, best_dice = [], 1, -1.0
if RESUME and LAST_FILE.exists():
    ck = torch.load(LAST_FILE, map_location=DEVICE)
    model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
    if ck["scaler"]:   # empty when the checkpoint was written without AMP (CPU)
        scaler.load_state_dict(ck["scaler"])
    start_epoch, best_dice = ck["epoch"] + 1, ck["best_dice"]
    log = pd.read_csv(LOG_FILE).to_dict("records") if LOG_FILE.exists() else []
    print(f"resumed from epoch {ck['epoch']} (best val Dice so far {best_dice:.4f})")

print(f"training {MODEL_NAME} ({'no' if AUGMENT == 'none' else AUGMENT} augmentation): epochs {start_epoch}-{MAX_EPOCHS}, {VOLUMES_PER_EPOCH} volumes per epoch")
for epoch in range(start_epoch, MAX_EPOCHS + 1):
    lr = LR * (1 - (epoch - 1) / MAX_EPOCHS) ** 0.9
    for g in opt.param_groups:
        g["lr"] = lr
    t0 = time.time(); tr_loss = train_one_epoch(epoch)
    row = {"epoch": epoch, "lr": lr, "train_loss": tr_loss, "val_dice": np.nan, "val_lesion_recall": np.nan}
    improved = False
    if epoch % VAL_EVERY == 0 or epoch == MAX_EPOCHS:
        vd, vr = validate(model, val_recs); row.update(val_dice=vd, val_lesion_recall=vr)
        if vd > best_dice:
            best_dice, improved = vd, True; torch.save({"model": model.state_dict(), "epoch": epoch, "val_dice": vd}, BEST_FILE)
    row["minutes"] = (time.time() - t0) / 60; log.append(row); pd.DataFrame(log).to_csv(LOG_FILE, index=False)
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "scaler": scaler.state_dict(), "epoch": epoch, "best_dice": best_dice}, LAST_FILE)
    msg = f"epoch {epoch:3d}/{MAX_EPOCHS} | loss {tr_loss:.3f} | lr {lr:.1e}"
    if row["val_dice"] == row["val_dice"]:
        msg += f" | val Dice {row['val_dice']:.3f} | val lesion recall {row['val_lesion_recall']:.2f}" + (" (best)" if improved else "")
    print(msg + f" | {row['minutes']:.1f} min"); status(msg)

if not BEST_FILE.exists():   # never validated (very short runs): keep the last weights
    torch.save({"model": model.state_dict(), "epoch": MAX_EPOCHS, "val_dice": float("nan")}, BEST_FILE)
lg = pd.read_csv(LOG_FILE)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(lg.epoch, lg.train_loss); ax[0].set_title("training loss"); ax[0].set_xlabel("epoch")
v = lg.dropna(subset=["val_dice"]); ax[1].plot(v.epoch, v.val_dice, "o-", label="val Dice"); ax[1].plot(v.epoch, v.val_lesion_recall, "s-", label="val lesion recall")
ax[1].set_title("clean validation (lung cases first)"); ax[1].set_xlabel("epoch"); ax[1].legend(); fig.tight_layout(); fig.savefig(FIG_DIR / "training_curves.png", dpi=120); plt.show()

## 9. Evaluation under the frozen perturbations

The best checkpoint is evaluated **without retraining** on every frozen condition: `test_lung` (Dice, lesion recall) and `test_neg` (false positives). Rows are appended to `per_case_metrics.csv` and `per_lesion.csv` after every study, so an interrupted run resumes where it stopped. Predictions for the first four lung test studies are saved for the overlay figures.

In [ ]:
ck = torch.load(BEST_FILE, map_location=DEVICE); model.load_state_dict(ck["model"]); model.eval()
print(f"evaluating best checkpoint (epoch {ck['epoch']}, val Dice {ck.get('val_dice', float('nan')):.4f})")

CASE_CSV, LESION_CSV, FEAT_CSV = OUT_DIR / "per_case_metrics.csv", OUT_DIR / "per_lesion.csv", OUT_DIR / "lesion_features.csv"
PRED_DIR = OUT_DIR / "predictions"; PRED_DIR.mkdir(exist_ok=True)
QUAL_CASES = [r["case_id"] for r in RECORDS["test_lung"][:4]]

def _append(df, path):
    df.to_csv(path, mode="a", header=not path.exists(), index=False)

feat_done = set(pd.read_csv(FEAT_CSV).case_id) if FEAT_CSV.exists() else set()

def run_eval(split_name):
    recs = RECORDS[split_name][:EVAL_MAX_CASES] if EVAL_MAX_CASES else RECORDS[split_name]
    done = set()
    if CASE_CSV.exists():
        d = pd.read_csv(CASE_CSV); done = set(zip(d.case_id, d.condition))
    for n_done, (rec, s) in enumerate(zip(recs, prefetch_map(lambda r: load_case(r, keep_raw=True), recs, 2, ahead=2)), 1):
        t_case = time.time()
        todo = [sp for sp in SPECS if (rec["case_id"], sp.name) not in done]
        if not todo:
            continue
        ct, pet = s["ct_raw"][0].numpy(), s["pet_raw"][0].numpy(); gt = s["mask"][0].numpy() > 0
        body = body_mask(ct); bb = bbox_slices(body, gt)
        gp = make_gt_pack(gt[bb], rec["voxel_ml"])
        if split_name == "test_lung" and rec["case_id"] not in feat_done:
            feats = pd.DataFrame(lesion_feature_rows(gp, pet[bb], ct[bb], body[bb])); feats.insert(0, "case_id", rec["case_id"])
            if len(feats):
                _append(feats, FEAT_CSV)
            feat_done.add(rec["case_id"])
        del pet, ct, body
        case_rows, lesion_rows = [], []
        for sp in todo:
            t0 = time.time()
            img = apply_spec(sp, s["image"], rec["case_id"])
            pred = predict_crop(model, img[(slice(None),) + bb]); del img
            row, lesions = score_prediction(pred, gp)
            case_rows.append({"run": RUN_NAME, "split": split_name, "case_id": rec["case_id"], "patient_id": rec["patient_id"],
                              "condition": sp.name, "family": sp.family, "level": sp.level, **row, "seconds": round(time.time() - t0, 1)})
            lesion_rows += [{"run": RUN_NAME, "case_id": rec["case_id"], "condition": sp.name, **l} for l in lesions]
            if rec["case_id"] in QUAL_CASES:
                np.savez_compressed(PRED_DIR / f"{hashlib.md5(rec['case_id'].encode()).hexdigest()[:10]}_{sp.name}.npz",
                                    pred=np.packbits(pred), crop_shape=np.array(pred.shape), bb=np.array([[x.start, x.stop] for x in bb]))
        _append(pd.DataFrame(case_rows), CASE_CSV)
        if lesion_rows:
            _append(pd.DataFrame(lesion_rows), LESION_CSV)
        c0 = next((r for r in case_rows if r["condition"] == "clean"), None)
        note = "" if c0 is None else (f"clean Dice {c0['dice']:.3f}" if c0["dice"] == c0["dice"] else f"clean false positives {c0['fp_volume_ml']:.1f} ml")
        msg = f"[{split_name} {n_done}/{len(recs)}] patient {rec['patient_id']} | {note} | {(time.time() - t_case) / 60:.1f} min"
        print(msg); status(msg)

t_start = time.time()
for sname in EVAL_SPLITS:
    run_eval(sname)
print(f"evaluation done in {(time.time()-t_start)/60:.1f} min -> {CASE_CSV}")

## 10. Results: tables, severity curves, overlays and saved metadata

Confidence intervals are 95% **patient-level bootstrap** intervals (patients, not studies or lesions, are resampled). Lesion recall is pooled over lesions (detected / total).

In [ ]:
def cluster_boot(df, num, den=None, n_boot=2000, seed=0):
    """Mean (or pooled ratio) with a patient-level bootstrap 95% CI. num/den are column names or Series."""
    d = pd.DataFrame({"p": df["patient_id"].values})
    d["n"] = df[num].values if isinstance(num, str) else np.asarray(num, float)
    d["d"] = df[den].values if isinstance(den, str) else (np.ones(len(df)) if den is None else np.asarray(den, float))
    d.loc[d["n"].isna(), ["n", "d"]] = 0.0           # undefined cases (e.g. Dice without lesions) do not count
    g = d.groupby("p")[["n", "d"]].sum(); N, D = g["n"].values, g["d"].values
    if D.sum() == 0:
        return np.nan, np.nan, np.nan
    pt = N.sum() / D.sum(); rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(g), (n_boot, len(g))); den_b = D[idx].sum(1)
    stat = np.where(den_b > 0, N[idx].sum(1) / np.where(den_b > 0, den_b, 1), np.nan)
    return pt, np.nanpercentile(stat, 2.5), np.nanpercentile(stat, 97.5)

def summarise(case_df):
    rows = []
    for (split_name, cond), g in case_df.groupby(["split", "condition"], sort=False):
        r = {"split": split_name, "condition": cond, "family": g.family.iloc[0], "level": g.level.iloc[0], "n_studies": len(g), "n_patients": g.patient_id.nunique()}
        if split_name == "test_lung":
            defined = lambda col: g[col].notna().astype(float).values
            for name, (num, den) in {"dice": ("dice", defined("dice")), "voxel_recall": ("voxel_recall", defined("voxel_recall")),
                                     "lesion_recall": ("n_detected", "n_gt_lesions"), "fp_volume_ml": ("fp_volume_ml", None)}.items():
                pt, lo, hi = cluster_boot(g, num, den)
                r.update({name: pt, name + "_lo": lo, name + "_hi": hi})
        else:   # lesion-free studies: only false positives are meaningful
            pt, lo, hi = cluster_boot(g, "fp_volume_ml"); r.update(fp_volume_ml=pt, fp_volume_ml_lo=lo, fp_volume_ml_hi=hi)
            pt, lo, hi = cluster_boot(g, (g.n_fp_components > 0).astype(float)); r.update(frac_with_fp=pt, frac_with_fp_lo=lo, frac_with_fp_hi=hi)
        rows.append(r)
    return pd.DataFrame(rows)

def _err(y, lo, hi):
    return np.clip([np.asarray(y) - np.asarray(lo), np.asarray(hi) - np.asarray(y)], 0, None)

def plot_severity(summary, metric, ylabel, ax_row, label=None, color=None, split_name="test_lung"):
    s = summary[summary.split == split_name]; clean = s[s.condition == "clean"].iloc[0]
    def pts(fam, levels):
        sub = s[s.family == fam].sort_values("level")
        x = [0] + list(sub.level); y = [clean[metric]] + list(sub[metric]); lo = [clean[metric + "_lo"]] + list(sub[metric + "_lo"]); hi = [clean[metric + "_hi"]] + list(sub[metric + "_hi"])
        return np.array(x), np.array(y), np.array(lo), np.array(hi)
    x, y, lo, hi = pts("lowres", None); x = np.where(x == 0, 1, x)
    ax_row[0].errorbar(x, y, yerr=_err(y, lo, hi), marker="o", capsize=3, label=label, color=color)
    ax_row[0].set_xlabel("low-resolution factor (1 = clean)"); ax_row[0].set_ylabel(ylabel); ax_row[0].set_title("blur / low resolution")
    for k, (fam, mk) in enumerate([("noise_pet", "o"), ("noise_ct", "s"), ("noise_both", "^")]):
        x, y, lo, hi = pts(fam, None)
        ax_row[1].errorbar(x + (k - 1) * 0.004, y, yerr=_err(y, lo, hi), marker=mk, capsize=3, ls=["-", "--", ":"][k], color=color, label=f"{label} / {fam[6:]}" if label else fam[6:])
    ax_row[1].set_xlabel("noise sigma (normalised units)"); ax_row[1].set_title("Gaussian noise on PET / CT / both")
    pr = s[s.condition == "pet_removed"].iloc[0]
    ax_row[2].errorbar([0, 1], [clean[metric], pr[metric]], yerr=_err([clean[metric], pr[metric]], [clean[metric + "_lo"], pr[metric + "_lo"]], [clean[metric + "_hi"], pr[metric + "_hi"]]),
                       marker="o", capsize=3, color=color, label=label)
    ax_row[2].set_xticks([0, 1]); ax_row[2].set_xticklabels(["PET present", "PET removed"]); ax_row[2].set_title("missing modality")
    for a in ax_row:
        a.grid(alpha=0.3)

case_df = pd.read_csv(CASE_CSV)
summary = summarise(case_df); summary.insert(0, "run", RUN_NAME)
summary.to_csv(OUT_DIR / "summary_by_condition.csv", index=False)
lung = summary[summary.split == "test_lung"]
if len(lung):
    c0, pr, worst = lung[lung.condition == "clean"].iloc[0], lung[lung.condition == "pet_removed"].iloc[0], lung.loc[lung.dice.idxmin()]
    print(f"Clean: Dice {c0.dice:.3f}, lesion recall {c0.lesion_recall:.3f} ({int(c0.n_patients)} lung patients)")
    print(f"PET removed: Dice {pr.dice:.3f} ({pr.dice - c0.dice:+.3f}), lesion recall {pr.lesion_recall:.3f}")
    print(f"Worst condition: {worst.condition} (Dice {worst.dice:.3f}, lesion recall {worst.lesion_recall:.3f})")
    ci = lambda r, k: f"{r[k]:.3f} [{r[k + '_lo']:.3f}, {r[k + '_hi']:.3f}]"
    display(pd.DataFrame({"Dice [95% CI]": [ci(r, "dice") for _, r in lung.iterrows()], "lesion recall [95% CI]": [ci(r, "lesion_recall") for _, r in lung.iterrows()],
                          "voxel recall": lung.voxel_recall.round(3).values, "false-positive ml / study": lung.fp_volume_ml.round(1).values},
                         index=pd.Index(lung.condition.values, name="condition")))
neg = summary[summary.split == "test_neg"]
if len(neg):
    print("\nLesion-free studies (false positives):")
    display(pd.DataFrame({"false-positive ml / study": neg.fp_volume_ml.round(2).values,
                          "studies with a false positive [95% CI]": [f"{r.frac_with_fp:.0%} [{r.frac_with_fp_lo:.0%}, {r.frac_with_fp_hi:.0%}]" for r in neg.itertuples()]},
                         index=pd.Index(neg.condition.values, name="condition")))

fig, axs = plt.subplots(2, 3, figsize=(14, 7.2), sharey="row")
plot_severity(summary, "dice", "Dice (lesion-positive lung studies)", axs[0], label=RUN_NAME, color="tab:blue")
plot_severity(summary, "lesion_recall", "lesion recall (pooled)", axs[1], label=RUN_NAME, color="tab:blue")
axs[0, 1].legend(fontsize=8)
fig.suptitle(f"{RUN_NAME}: lung test set, metrics vs perturbation severity (95% patient-level bootstrap CI)"); fig.tight_layout()
fig.savefig(FIG_DIR / "severity_curves.png", dpi=130); plt.show()

In [ ]:
def qualitative_overlays(conds=("clean", "lowres_x4", "noise_both_s0.1", "pet_removed"), n_cases=2):
    shown = 0
    for rec in RECORDS["test_lung"]:
        if rec["case_id"] not in QUAL_CASES or shown >= n_cases:
            continue
        tag = hashlib.md5(rec["case_id"].encode()).hexdigest()[:10]
        if not all((PRED_DIR / f"{tag}_{c}.npz").exists() for c in conds):
            continue
        s = load_case(rec); ct = s["image"][CT].numpy(); gt = s["mask"][0].numpy() > 0
        _, _, z = _slice_through_lesion(s["mask"])
        fig, axs = plt.subplots(1, len(conds), figsize=(4.2 * len(conds), 4.4))
        for a, c in zip(axs, conds):
            z_ = np.load(PRED_DIR / f"{tag}_{c}.npz"); cs = tuple(int(v) for v in z_["crop_shape"]); bb = tuple(slice(int(lo), int(hi)) for lo, hi in z_["bb"])
            pred = np.zeros(gt.shape, bool); pred[bb] = np.unpackbits(z_["pred"])[: int(np.prod(cs))].reshape(cs).astype(bool)
            shown_img = apply_spec(COND[c], s["image"], rec["case_id"])[PET, :, :, z].numpy()
            a.imshow(shown_img.T, cmap="gray_r", vmin=0, vmax=0.4, origin="lower")
            _contour(a, gt[:, :, z])
            if pred[:, :, z].any():
                a.contour(pred[:, :, z].T.astype(float), levels=[0.5], colors="red", linewidths=0.9, origin="lower")
            a.set_title(c, fontsize=10); a.axis("off")
        fig.suptitle(f"{RUN_NAME}: PET (as seen by the model), green = ground truth, red = prediction"); fig.tight_layout()
        fig.savefig(FIG_DIR / f"overlay_{tag}.png", dpi=120); plt.show(); shown += 1
qualitative_overlays()

# ---- save everything needed to reproduce / compare this run ----
meta = {"run": RUN_NAME, "model": MODEL_NAME, "augment": AUGMENT, "params_million": round(n_params / 1e6, 2), "patch": PATCH, "batch": BATCH,
        "patches_per_volume": PATCHES_PER_VOLUME, "filters": FILTERS, "max_epochs": MAX_EPOCHS, "volumes_per_epoch": VOLUMES_PER_EPOCH, "lr": LR,
        "seed": SEED, "best_epoch": int(ck["epoch"]), "split_fingerprint": split_fp, "perturbation_fingerprint": fp, "pet_clip": PET_CLIP,
        "ct_clip": CT_CLIP, "lesion_min_overlap": LESION_MIN_OVERLAP, "pred_threshold": PRED_THRESHOLD, "sw_overlap": SW_OVERLAP, "env": ENV,
        "finished": time.strftime("%Y-%m-%d %H:%M:%S")}
json.dump(meta, open(OUT_DIR / "config.json", "w"), indent=2, default=str)
(OUT_DIR / "COMPLETE").write_text(meta["finished"]); status("COMPLETE")
print(f"results saved to {OUT_DIR}")

## 11. Compare runs

Overlays every run found in `runs/` (your model and your teammate's) on the same axes, after checking that they used the same split and the same perturbations. To bring a teammate's results in: ask for the folder `runs/<their run>/` **without** the `.pt` checkpoint files and drop it into your `runs/` folder.

In [ ]:
def compare_runs(run_names=None, root=CODE_DIR / "runs"):
    runs = {}
    for d in sorted(root.glob("*")):
        if (d / "per_case_metrics.csv").exists() and (d / "config.json").exists() and (run_names is None or d.name in run_names):
            runs[d.name] = {"cfg": json.load(open(d / "config.json")), "cases": pd.read_csv(d / "per_case_metrics.csv")}
    if len(runs) == 0:
        print("no finished runs found"); return None
    if len(runs) < 2:
        print("only one finished run so far: train a second model (new RUN_NAME) and re-run this cell to compare"); return None
    ref = next(iter(runs.values()))["cfg"]
    for n, r in runs.items():
        ok = r["cfg"]["split_fingerprint"] == ref["split_fingerprint"] and r["cfg"]["perturbation_fingerprint"] == ref["perturbation_fingerprint"]
        print(f"{n:30s} model={r['cfg']['model']:10s} augment={r['cfg']['augment']:8s} studies={len(r['cases'])} comparable={'yes' if ok else 'NO: different split or perturbations'}")
    sums = {n: summarise(r["cases"]) for n, r in runs.items()}
    colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    fig, axs = plt.subplots(2, 3, figsize=(14, 7.2), sharey="row")
    for k, (n, s) in enumerate(sums.items()):
        plot_severity(s, "dice", "Dice (lung)", axs[0], label=n, color=colors[k % len(colors)])
        plot_severity(s, "lesion_recall", "lesion recall (lung, pooled)", axs[1], label=n, color=colors[k % len(colors)])
    axs[0, 0].legend(fontsize=8); fig.suptitle("lung test set: metrics vs perturbation severity, all runs"); fig.tight_layout(); plt.show()
    table = pd.concat([s[s.split == "test_lung"].set_index("condition")["dice"].rename(n) for n, s in sums.items()], axis=1)
    print("Dice per condition (lung test set):"); display(table.round(3))
    negs = [s[s.split == "test_neg"].set_index("condition")["fp_volume_ml"].rename(n) for n, s in sums.items() if (s.split == "test_neg").any()]
    if negs:
        print("false-positive volume (ml) on lesion-free studies:"); display(pd.concat(negs, axis=1).round(2))
    return sums

_ = compare_runs()

### Optional: package this run for sharing

Creates `runs/<RUN_NAME>_results.zip` with the metrics, figures and config but **without** checkpoints (small enough to send).

In [ ]:
import zipfile
zip_path = OUT_DIR.parent / f"{RUN_NAME}_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT_DIR.rglob("*"):
        if p.is_file() and p.suffix != ".pt":
            z.write(p, Path(RUN_NAME) / p.relative_to(OUT_DIR))
print("wrote", zip_path, f"({zip_path.stat().st_size/1e6:.1f} MB)")